# B — DR formula variant testing

The current preprocessing pipeline (`1-Data_preprocessing.ipynb`) computes the physics-based estimator `fwns_calculated` using the De Lannoy *Difference Ratio* formula:

$$
f_{\text{wns}} \;=\; \frac{\epsilon_{\text{land}}^{\text{ref}} - \epsilon_{\text{obs}}}{\epsilon_{\text{land}}^{\text{ref}} - \epsilon_{\text{water}}^{\text{ref}}}
$$

with a single fixed choice of frequency/polarization (`K_h` for both land slots, `19H` for the observed emissivity). Empirically this is not the best variant.

This notebook **systematically sweeps every combination** of the available frequency/polarization options for each emissivity slot and ranks the resulting estimators against the true `fwns` label, mirroring the baseline-evaluation style of `3-Model_training.ipynb` (Section 1).

**Sweep design**

The reference land emissivity LUT is only loaded for the K band in [1-Data_preprocessing.ipynb:140-141](1-Data_preprocessing.ipynb#L140) (`lut_de_lannoy_K_h.csv` / `lut_de_lannoy_K_v.csv`), so the land slots are restricted to K-band.

| Slot | Options | Count |
|---|---|---|
| Numerator land emissivity | `ref_land_emis_de_lannoy_K_{h,v}` | 2 |
| Denominator land emissivity | `ref_land_emis_de_lannoy_K_{h,v}` | 2 |
| Observed emissivity | `emiss{19,37}{H,V}_de_lannoy` | 4 |

→ **2 × 2 × 4 = 16 variants** (numerator and denominator land slots are swept independently).

**Open-water emissivity constant.** Only `REF_WATER_EMISS_H = 0.288760` is defined in the codebase. We reuse it for all 16 variants. The variant ranking is therefore measuring the effect of the land/obs swaps; the absolute metrics for V-pol-obs variants carry an additional caveat (the constant is not strictly the right one for V).

**Evaluation split.** Year 2018 (test split), matching the convention of notebook 3.

## 1. Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import product
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

DATA_DIR = Path("data/datasets")
TEST_YEAR = 2018
REF_WATER_EMISS_H = 0.288760

# Knob for fast iteration. Set to 1.0 for the final run.
SAMPLE_FRAC = 1.0
RANDOM_SEED = 0

# Reference land emissivity LUT is only loaded for K band in 1-Data_preprocessing.ipynb,
# so Ka-band ref_land variables do not exist in the parquet files.
LAND_COLS = [
    "ref_land_emis_de_lannoy_K_h",
    "ref_land_emis_de_lannoy_K_v",
]
OBS_COLS = [
    "emiss19H_de_lannoy",
    "emiss19V_de_lannoy",
    "emiss37H_de_lannoy",
    "emiss37V_de_lannoy",
]
NEEDED = sorted(set(LAND_COLS + OBS_COLS + ["fwns", "year"]))

# Short tag helpers used in variant names: "K_h" -> "Kh", "emiss19H" -> "19H"
def land_tag(col):
    parts = col.split("_")  # ['ref','land','emis','de','lannoy','K','h']
    band, pol = parts[-2], parts[-1]
    return f"{band}{pol}"

def obs_tag(col):
    return col.replace("emiss", "").replace("_de_lannoy", "")

print(f"Land options: {[land_tag(c) for c in LAND_COLS]}")
print(f"Obs options:  {[obs_tag(c) for c in OBS_COLS]}")
print(f"Total variants: {len(LAND_COLS) * len(LAND_COLS) * len(OBS_COLS)}")


## 2. Load the test split

Read only the columns we need from the 2018 daily parquet files and concatenate.

In [ ]:
test_files = sorted(DATA_DIR.glob(f"windsat_{TEST_YEAR}_*.parquet"))
print(f"Found {len(test_files)} parquet files for year {TEST_YEAR}")

frames = [pd.read_parquet(p, columns=NEEDED) for p in test_files]
df = pd.concat(frames, ignore_index=True)
del frames

# Keep only test rows (defensive — files are already split by year, but the column is there).
df = df[df["year"] == TEST_YEAR]

# Drop rows with no ground truth.
df = df.dropna(subset=["fwns"])

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Loaded {len(df):,} rows  |  memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(3)


## 3. Compute every variant of the DR formula

For each `(land_num, land_den, obs)` triple we compute

$$
\hat f = \frac{\epsilon_{\text{land}}^{\text{num}} - \epsilon_{\text{obs}}}{\epsilon_{\text{land}}^{\text{den}} - \text{REF\_WATER\_EMISS\_H}}
$$

Variant names follow the pattern `num=Kh|den=Kh|obs=19H` so we can group/sort by slot later. The original baseline is `num=Kh|den=Kh|obs=19H`.

In [ ]:
# Cache the 4 denominator series (one per land-den option) so we don't recompute them 16 times each.
denoms = {
    land_den: (df[land_den] - REF_WATER_EMISS_H).where(
        (df[land_den] - REF_WATER_EMISS_H).abs() > 1e-9
    )
    for land_den in LAND_COLS
}

variant_specs = []   # list of (name, land_num, land_den, obs)
preds = {}           # name -> Series

for land_num, land_den, obs in product(LAND_COLS, LAND_COLS, OBS_COLS):
    name = f"num={land_tag(land_num)}|den={land_tag(land_den)}|obs={obs_tag(obs)}"
    pred = (df[land_num] - df[obs]) / denoms[land_den]
    preds[name] = pred.astype(np.float32)
    variant_specs.append((name, land_num, land_den, obs))

preds_df = pd.DataFrame(preds, index=df.index)
print(f"Computed {preds_df.shape[1]} variant predictions over {len(preds_df):,} rows")

ORIGINAL_VARIANT = f"num={land_tag('ref_land_emis_de_lannoy_K_h')}|den={land_tag('ref_land_emis_de_lannoy_K_h')}|obs={obs_tag('emiss19H_de_lannoy')}"
print(f"Original variant tag: {ORIGINAL_VARIANT}")


## 4. Compute metrics for every variant

Same metric block as `3-Model_training.ipynb`, Section 1: MAE, RMSE, R², bias. We additionally report:

- `MAE_clip / RMSE_clip / R2_clip`: same metrics on `pred.clip(0, 1)` — useful because the raw DR ratio can fall outside `[0, 1]`.
- `frac_in_range`: fraction of predictions that already fall within `[0, 1]`.

NaN/inf rows are dropped pairwise per variant before computing.

In [ ]:
y_true_full = df["fwns"].to_numpy()

def pol_of_land(land_col):
    return land_col[-1].upper()  # 'h' or 'v' -> 'H'/'V'

rows = []
for name, land_num, land_den, obs in variant_specs:
    pred = preds_df[name].to_numpy()
    mask = np.isfinite(pred) & np.isfinite(y_true_full)
    yt = y_true_full[mask]
    yp = pred[mask]

    mae  = mean_absolute_error(yt, yp)
    rmse = root_mean_squared_error(yt, yp)
    r2   = r2_score(yt, yp)
    bias = float((yp - yt).mean())

    yp_c = np.clip(yp, 0.0, 1.0)
    mae_c  = mean_absolute_error(yt, yp_c)
    rmse_c = root_mean_squared_error(yt, yp_c)
    r2_c   = r2_score(yt, yp_c)

    frac_in_range = float(((yp >= 0.0) & (yp <= 1.0)).mean())

    rows.append({
        "variant": name,
        "land_num": land_tag(land_num),
        "land_den": land_tag(land_den),
        "obs": obs_tag(obs),
        "land_num_pol": pol_of_land(land_num),
        "land_den_pol": pol_of_land(land_den),
        "obs_freq": obs_tag(obs)[:2],   # '19' or '37'
        "obs_pol":  obs_tag(obs)[-1],   # 'H' or 'V'
        "tied_land": land_num == land_den,
        "MAE":  mae,  "RMSE":  rmse,  "R2":  r2,  "bias": bias,
        "MAE_clip": mae_c, "RMSE_clip": rmse_c, "R2_clip": r2_c,
        "frac_in_range": frac_in_range,
        "n_valid": int(mask.sum()),
    })

results_df = pd.DataFrame(rows).set_index("variant")
print(f"results_df: {results_df.shape}")
results_df.head(3)


## 5. Ranking and sanity check

Sort all 64 variants by RMSE. The original baseline (`num=Kh|den=Kh|obs=19H`) is highlighted as the reference.

**Sanity check.** Notebook 3's physics baseline reported approximately:

| Metric | Value |
|---|---|
| MAE  | 0.0346 |
| RMSE | 0.0531 |
| R²   | 0.776  |
| bias | 0.0095 |

Our `ORIGINAL_VARIANT` row should match these values (modulo small differences from any `dropna(subset=['fwns'])` choice). A noticeable deviation indicates a column-selection or NaN-masking bug above.

In [ ]:
ranked = results_df.sort_values("RMSE")

print("=== Original baseline ===")
print(results_df.loc[[ORIGINAL_VARIANT], ["MAE", "RMSE", "R2", "bias", "frac_in_range"]].T)

print("\n=== All 16 variants ranked by RMSE ===")
display_cols = ["MAE", "RMSE", "R2", "bias", "frac_in_range",
                "MAE_clip", "RMSE_clip", "R2_clip"]
print(ranked[display_cols].round(5).to_string())

best = ranked.index[0]
orig = results_df.loc[ORIGINAL_VARIANT]
top  = results_df.loc[best]
print("\n=== Best vs Original ===")
print(f"Best     : {best}")
print(f"           MAE={top['MAE']:.5f}  RMSE={top['RMSE']:.5f}  R2={top['R2']:.5f}  bias={top['bias']:+.5f}")
print(f"Original : {ORIGINAL_VARIANT}")
print(f"           MAE={orig['MAE']:.5f}  RMSE={orig['RMSE']:.5f}  R2={orig['R2']:.5f}  bias={orig['bias']:+.5f}")
print(f"Δ RMSE   : {top['RMSE'] - orig['RMSE']:+.5f}  ({(top['RMSE']/orig['RMSE'] - 1)*100:+.1f}%)")


## 6. Tied-land subset — 2×4 heatmap

The 8 variants where `land_num == land_den` are the physically coherent subset (the formula is derived from a two-endmember mixing model with a single land endmember). The heatmap below shows RMSE / MAE / R² for each `(land, obs)` combination — this is the headline figure for choosing a replacement for `fwns_calculated`.

In [ ]:
tied = results_df[results_df["tied_land"]].copy()
tied["land"] = tied["land_num"]

land_order = ["Kh", "Kv"]
obs_order  = ["19H", "19V", "37H", "37V"]

heat_rmse = tied.pivot(index="land", columns="obs", values="RMSE").reindex(index=land_order, columns=obs_order)
heat_mae  = tied.pivot(index="land", columns="obs", values="MAE" ).reindex(index=land_order, columns=obs_order)
heat_r2   = tied.pivot(index="land", columns="obs", values="R2"  ).reindex(index=land_order, columns=obs_order)

fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))

sns.heatmap(heat_rmse, annot=True, fmt=".4f", cmap="viridis_r", ax=axes[0], cbar_kws={"label": "RMSE"})
axes[0].set_title("RMSE — tied-land variants (lower = better)")
axes[0].set_xlabel("observed emissivity")
axes[0].set_ylabel("land emissivity (LUT)")

sns.heatmap(heat_mae, annot=True, fmt=".4f", cmap="viridis_r", ax=axes[1], cbar_kws={"label": "MAE"})
axes[1].set_title("MAE — tied-land variants")
axes[1].set_xlabel("observed emissivity")
axes[1].set_ylabel("")

sns.heatmap(heat_r2, annot=True, fmt=".3f", cmap="viridis", ax=axes[2], cbar_kws={"label": "R²"})
axes[2].set_title("R² — tied-land variants (higher = better)")
axes[2].set_xlabel("observed emissivity")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()


## 7. Full 16-variant ranking — bar plot

Bars are sorted by RMSE; bar colors mark whether `land_num == land_den` (i.e. the tied-vs-untied subset). The original baseline is annotated with a red marker. This view answers two questions:

1. Are any untied combinations meaningfully better than the best tied one?
2. Where does the original variant rank among the 16?

In [ ]:
ranked = results_df.sort_values("RMSE")
colors = np.where(ranked["tied_land"], "#1f77b4", "#cccccc")

fig, ax = plt.subplots(figsize=(10, 6))
ypos = np.arange(len(ranked))
ax.barh(ypos, ranked["RMSE"].values, color=colors, edgecolor="none")
ax.set_yticks(ypos)
ax.set_yticklabels(ranked.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("RMSE")
ax.set_title("All 16 DR variants ranked by RMSE  (blue = tied land slots, grey = untied)")

orig_pos = ranked.index.get_loc(ORIGINAL_VARIANT)
ax.axhline(orig_pos, color="red", linestyle=":", linewidth=1, alpha=0.6)
ax.scatter([ranked.iloc[orig_pos]["RMSE"]], [orig_pos], color="red", zorder=5,
           label=f"Original (rank {orig_pos + 1}/{len(ranked)})")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Distribution comparison — truth vs original / best / worst

Histograms (KDE-overlaid) of the predicted `fwns` for three reference variants compared with the ground-truth distribution. A good variant should reproduce both the spike near 0 (mostly-land pixels) and the heavy upper tail toward 1.

In [ ]:
best_variant  = ranked.index[0]
worst_variant = ranked.index[-1]

picks = [
    ("Original", ORIGINAL_VARIANT, "#7f7f7f"),
    ("Best",     best_variant,     "#2ca02c"),
    ("Worst",    worst_variant,    "#d62728"),
]

bins = np.linspace(-0.2, 1.2, 80)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
y_true_arr = df["fwns"].to_numpy()

for ax, (label, variant, color) in zip(axes, picks):
    pred = preds_df[variant].to_numpy()
    pred = pred[np.isfinite(pred)]
    ax.hist(y_true_arr, bins=bins, density=True, alpha=0.45, color="#1f77b4", label="fwns (truth)")
    ax.hist(pred,        bins=bins, density=True, alpha=0.55, color=color,    label=f"{label}: {variant.split('|')[-1]}")
    ax.axvline(0.0, color="black", linewidth=0.6, alpha=0.5)
    ax.axvline(1.0, color="black", linewidth=0.6, alpha=0.5)
    m = results_df.loc[variant]
    ax.set_title(f"{label}  RMSE={m['RMSE']:.4f}  R²={m['R2']:.3f}\n{variant}", fontsize=9)
    ax.set_xlabel("fwns")
    ax.legend(loc="upper center", fontsize=8)

axes[0].set_ylabel("density")
plt.tight_layout()
plt.show()


## 9. Residual histograms — original vs best

Same visual style as the physics-baseline diagnostics in `3-Model_training.ipynb`, Section 1: histogram of residuals (predicted − truth) with a vertical line at 0 and the bias annotated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
res_bins = np.linspace(-1.0, 1.0, 80)

for ax, (label, variant, color) in zip(axes, [("Original", ORIGINAL_VARIANT, "steelblue"),
                                              ("Best",     best_variant,     "#2ca02c")]):
    pred = preds_df[variant].to_numpy()
    mask = np.isfinite(pred)
    residuals = pred[mask] - y_true_arr[mask]
    bias = float(residuals.mean())
    rmse = results_df.loc[variant, "RMSE"]

    ax.hist(residuals, bins=res_bins, color=color, edgecolor="none", alpha=0.85)
    ax.axvline(0.0, color="red", linestyle="--", linewidth=1)
    ax.axvline(bias, color="black", linestyle=":", linewidth=1, label=f"mean = {bias:+.4f}")
    ax.set_title(f"{label} residuals  (RMSE={rmse:.4f}, bias={bias:+.4f})\n{variant}", fontsize=9)
    ax.set_xlabel("residual (predicted − truth)")
    ax.legend()

axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()


## 10. Scatter plots — predicted vs truth

5 000-row random samples per panel (same approach as `3-Model_training.ipynb`'s baseline scatter). Closeness to the 1:1 line indicates a good DR variant.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
sample_size = min(5000, len(df))
sample_idx = rng.choice(len(df), size=sample_size, replace=False)
y_sample = y_true_arr[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, (label, variant, color) in zip(axes, [("Original", ORIGINAL_VARIANT, "steelblue"),
                                              ("Best",     best_variant,     "#2ca02c")]):
    pred_sample = preds_df[variant].to_numpy()[sample_idx]
    m = results_df.loc[variant]
    ax.scatter(y_sample, pred_sample, alpha=0.3, s=6, color=color)
    ax.plot([0, 1], [0, 1], "r--", linewidth=1.4, label="1:1 line")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.6, 1.6)
    ax.set_xlabel("fwns (truth)")
    ax.set_ylabel("predicted fwns")
    ax.set_title(f"{label}\n{variant}", fontsize=9)
    ax.text(0.02, 0.98,
            f"MAE  = {m['MAE']:.4f}\nRMSE = {m['RMSE']:.4f}\nR²   = {m['R2']:.3f}\nbias = {m['bias']:+.4f}",
            transform=ax.transAxes, va="top", ha="left",
            fontsize=9, family="monospace",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="lightgrey"))
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


## 11. Group-level summaries

Average each metric over all variants that share a given slot value. This separates the *driver* of any improvement: is it polarization, frequency, or whether the land slots are tied?

In [ ]:
metric_cols = ["MAE", "RMSE", "R2", "bias"]

print("=== Mean metrics by observed-emissivity polarization ===")
print(results_df.groupby("obs_pol")[metric_cols].mean().round(5).to_string())

print("\n=== Mean metrics by observed-emissivity frequency ===")
print(results_df.groupby("obs_freq")[metric_cols].mean().round(5).to_string())

print("\n=== Mean metrics by numerator land polarization ===")
print(results_df.groupby("land_num_pol")[metric_cols].mean().round(5).to_string())

print("\n=== Mean metrics: tied vs untied land slots ===")
print(results_df.groupby("tied_land")[metric_cols].mean().round(5).to_string())

print("\n=== Mean metrics by polarization match (land_num vs obs) ===")
results_df_aug = results_df.assign(pol_match=(results_df["land_num_pol"] == results_df["obs_pol"]))
print(results_df_aug.groupby("pol_match")[metric_cols].mean().round(5).to_string())


## 12. Conclusion

Read the printed comparison in **Section 5** ("Best vs Original") and the heatmap in **Section 6** to identify the winning variant.

If the best tied-land variant materially improves over the original `(num=Kh|den=Kh|obs=19H)`, the recommendation is to update the formula in `1-Data_preprocessing.ipynb` at the two locations where `fwns_calculated` is computed and re-export the parquet dataset:

- [1-Data_preprocessing.ipynb:2989](1-Data_preprocessing.ipynb#L2989) — augmented dataset cell
- [1-Data_preprocessing.ipynb:3343](1-Data_preprocessing.ipynb#L3343) — per-day pipeline cell

The same change should also be reflected in the `denominator`, `term_1`, `term_2` derived columns (lines [2986-2988](1-Data_preprocessing.ipynb#L2986) and [3340-3342](1-Data_preprocessing.ipynb#L3340)), since notebook 3 uses them as features in the model-training feature blocks.

**Caveat to keep in mind.** The open-water emissivity constant `REF_WATER_EMISS_H = 0.288760` is strictly valid for 18.7 GHz, H polarization. Variants whose denominator pairs a 37 GHz / V-pol land emissivity with this H-pol water constant are physically inconsistent. They may still rank well empirically (because the constant is buried inside a difference and acts mostly as an offset/scale), but a future iteration of this analysis should source matching `REF_WATER_EMISS_{V, Ka_h, Ka_v}` values from De Lannoy et al. 2016 Table 2 and re-run the sweep.